# Import setting

In [ ]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

%load_ext autoreload
%autoreload 2

from src.config import Configuration

CONFIG = Configuration(
    batch_size=32, # que sino no caben los 3
)

/home/turbotowerlnx/Documents/Master/VPC/VPC-Labs/notebooks
/home/turbotowerlnx/Documents/Master/VPC/VPC-Labs/app


# Load data

In [2]:
from src.data import load_gender_data

train_loader, test_loader = load_gender_data(CONFIG)

# Load models

In [3]:
import re
import torch
from pathlib import Path

from src.models import GenderModule


top_models = [
    "gender-best-epoch=38-val_acc=0.9577.ckpt",
    "gender-best-epoch=43-val_acc=0.9592.ckpt",
    "gender-best-epoch=46-val_acc=0.9588.ckpt"
]

loaded_models = []
for ckpt in top_models:
    lightning_model = GenderModule.load_from_checkpoint(
        os.path.join(CONFIG.MODELS_PATH, ckpt),
        CONFIG=CONFIG,
        map_location=CONFIG.device,
    )
    lightning_model.eval()
    lightning_model.to(CONFIG.device)
    loaded_models.append(lightning_model)

print(f"Loaded {len(loaded_models)} models")

Loaded 3 models


In [ ]:
@torch.no_grad()
def evaluate_model(lightning_model, data_loader, device):
    correct = 0
    total = 0
    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = lightning_model(images)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()
    return correct / total if total > 0 else 0.0

# Individual model accuracies
individual_accs = []
for i, m in enumerate(loaded_models, start=1):
    acc = evaluate_model(m, test_loader, CONFIG.device)
    individual_accs.append(acc)
    print(f"model_{i} accuracy: {acc:.4f}")

# Soft-voting ensemble accuracy
ensemble_correct = 0
ensemble_total = 0
for images, labels in test_loader:
    images = images.to(CONFIG.device)
    labels = labels.to(CONFIG.device)

    probs_sum = None
    for m in loaded_models:
        logits = m(images)
        probs = torch.softmax(logits, dim=1)
        probs_sum = probs if probs_sum is None else probs_sum + probs

    # ensemble_preds = torch.argmax(probs_sum, dim=1)
    # ensemble_correct += (ensemble_preds == labels).sum().item()
    # ensemble_total += labels.numel()
    probs_avg = probs_sum / len(loaded_models)
    ensemble_preds = torch.argmax(probs_avg, dim=1)
    ensemble_correct += (ensemble_preds == labels).sum().item()
    ensemble_total += labels.numel()

ensemble_acc = ensemble_correct / ensemble_total if ensemble_total > 0 else 0.0
print(f"ensemble accuracy: {ensemble_acc:.4f}")

SyntaxError: invalid syntax (4078794419.py, line 37)